# First dive: *Le Petit Prince* MEG listening dataset

## Introduction

This dataset (OpenNeuro `ds007523`) contains **magnetoencephalography (MEG)** recordings
of participants listening to the French audiobook *Le Petit Prince*, by Antoine de
Saint-Exupéry. It is analyzed in d'Ascoli et al., *Towards decoding individual words
from non-invasive brain recordings*, Nature Communications 16, 10521 (2025).

A complementary MEG dataset using a reading (RSVP) paradigm from the same project is
available as `ds007524`.

**Key facts:**

- **Participants**: 58 healthy adults (17 female, mean age 27.8 ± 5.5 years), all
  native French speakers, right-handed, no neurological disorders.
- **Stimulus**: the audiobook, split into 9 parts (~10min50s ± 55s each, range
  9min40s–12min30s).
- **Task**: participants passively listened to one audiobook part per run (9 runs
  total), with 4 comprehension questions between runs and short breaks.
- **MEG system**: Elekta Neuromag TRIUX (102 magnetometers + 204 planar
  gradiometers), sampled at 1000 Hz, online band-pass 0.1–330 Hz. EOG/ECG were
  recorded alongside for artifact monitoring.
- **Anatomy**: a T1-weighted MRI per participant (3T Siemens Prisma) for
  coregistration/source analysis.
- Data follow the **BIDS/MEG-BIDS** standard.

The goal of this notebook is just to get familiar with the dataset's structure and
contents (participants, files, basic MEG file info) — not to perform any analysis.


In [1]:
import sys
sys.path.append('..')

import json
import pandas as pd
import mne

from config import LISTEN_DIR

LISTEN_DIR

PosixPath('/mnt/sda/data-output/phd/Pallier2025/listen/ds007523-download')

## Participants

The `participants.tsv` file at the dataset root lists demographics for all 58
subjects.


In [2]:
participants = pd.read_csv(LISTEN_DIR / 'participants.tsv', sep='\t')
print(f"{len(participants)} participants")
participants.describe(include='all')

58 participants


,participant_id,age,sex,hand
count,58,58.000000,58,58
unique,58,NaN,2,1
top,sub-01,NaN,M,R
freq,1,NaN,39,58
mean,NaN,27.793103,NaN,NaN
std,NaN,5.479655,NaN,NaN
min,NaN,18.000000,NaN,NaN
25%,NaN,23.000000,NaN,NaN
50%,NaN,28.500000,NaN,NaN
75%,NaN,32.000000,NaN,NaN


## Data organization and complete file-content preview

Each subject has a `ses-01/` session with MEG recordings and, when available, an anatomical MRI. The MEG folder contains up to nine listening runs. A typical run has four files: the MEG recording (`*_meg.fif`), its JSON metadata, `events.tsv`, and `channels.tsv`; session-level MEG calibration, crosstalk, and coordinate-system files are also present. The shared sourcedata contains stimulus materials and run-level extra information.

The inventory below previews every file found under `LISTEN_DIR`, including every subject and run. Tables show their first five rows, text and JSON show an opening excerpt, and binary files show readable metadata or header bytes where possible. This keeps the actual content visible while making it clear which files are text-readable and which are binary recordings or images.

In [3]:
import re

all_files = sorted(path for path in LISTEN_DIR.rglob('*') if path.is_file())
subject_dirs = sorted(path for path in LISTEN_DIR.glob('sub-*') if path.is_dir())
subject_file_sets = {
    subject.name: {
        path.relative_to(LISTEN_DIR).as_posix()
        for path in subject.rglob('*')
        if path.is_file()
    }
    for subject in subject_dirs
}

dataset_level_files = [
    path.relative_to(LISTEN_DIR).as_posix()
    for path in all_files
    if not path.relative_to(LISTEN_DIR).parts[0].startswith('sub-')
]
print(f"{len(all_files):,} files across {len(subject_dirs)} subjects")
print(f"\nDataset-level files ({len(dataset_level_files)}):")
print('\n'.join(dataset_level_files))

representative_subject = subject_dirs[0]
print(f"\nComplete subject example: {representative_subject.name}")
for path in sorted(representative_subject.rglob('*')):
    if path.is_file():
        print(path.relative_to(LISTEN_DIR).as_posix())

normalized_paths = {
    subject: {re.sub(r'sub-\d+', 'sub-XX', path) for path in paths}
    for subject, paths in subject_file_sets.items()
}
all_subject_patterns = sorted(set.union(*normalized_paths.values()))
pattern_rows = []
for pattern in all_subject_patterns:
    present = sorted(
        subject for subject, paths in normalized_paths.items() if pattern in paths
    )
    example_path = next(
        path for path in subject_file_sets[present[0]]
        if re.sub(r'sub-\d+', 'sub-XX', path) == pattern
    )
    pattern_rows.append({
        'subject path pattern': pattern,
        'subjects containing file': len(present),
        'example path': example_path,
    })
print('\nSubject-specific path patterns (shared layouts shown once):')
with pd.option_context('display.max_colwidth', None):
    display(pd.DataFrame(pattern_rows))

missing_rows = []
for pattern in all_subject_patterns:
    missing = sorted(
        subject for subject, paths in normalized_paths.items() if pattern not in paths
    )
    if missing:
        missing_rows.append({
            'path pattern': pattern,
            'missing subjects': ', '.join(missing),
        })
print('\nExceptions to the shared subject layout:')
if missing_rows:
    with pd.option_context('display.max_colwidth', None):
        display(pd.DataFrame(missing_rows))
else:
    print('No missing subject paths.')

coverage_rows = []
for subject in subject_dirs:
    subject_meg_dir = subject / 'ses-01' / 'meg'
    complete_runs = []
    incomplete_runs = []
    for run in range(1, 10):
        prefix = f"{subject.name}_ses-01_task-listen_run-{run:02d}"
        required_files = [
            subject_meg_dir / f'{prefix}{suffix}'
            for suffix in ('_channels.tsv', '_events.tsv', '_meg.fif', '_meg.json')
        ]
        if all(path.is_file() for path in required_files):
            complete_runs.append(run)
        else:
            missing_suffixes = [
                path.name.removeprefix(prefix)
                for path in required_files
                if not path.is_file()
            ]
            incomplete_runs.append(
                f"{run:02d} (missing {', '.join(missing_suffixes)})"
            )
    coverage_rows.append({
        'subject': subject.name,
        'files': len(subject_file_sets[subject.name]),
        'T1w MRI': (subject / 'ses-01' / 'anat' / f'{subject.name}_ses-01_T1w.nii.gz').is_file(),
        'complete runs': len(complete_runs),
        'incomplete runs': '; '.join(incomplete_runs) or '—',
    })
print('\nFile and recording coverage by subject:')
display(pd.DataFrame(coverage_rows))

2,506 files across 58 subjects

Dataset-level files (91):
.datalad/config
.gitattributes
CHANGES
LICENSE
README
datacite.yml
dataset_description.json
participants.json
participants.tsv
sourcedata/stimuli/audio/ch1-3.wav
sourcedata/stimuli/audio/ch10-12.wav
sourcedata/stimuli/audio/ch13-14.wav
sourcedata/stimuli/audio/ch15-19.wav
sourcedata/stimuli/audio/ch20-22.wav
sourcedata/stimuli/audio/ch23-25.wav
sourcedata/stimuli/audio/ch26-27.wav
sourcedata/stimuli/audio/ch4-6.wav
sourcedata/stimuli/audio/ch7-9.wav
sourcedata/stimuli/data_path.txt
sourcedata/stimuli/offset_stretch/infos.csv
sourcedata/stimuli/origin.txt
sourcedata/stimuli/phonemes/ch1-3.TextGrid
sourcedata/stimuli/phonemes/ch10-12.TextGrid
sourcedata/stimuli/phonemes/ch13-14.TextGrid
sourcedata/stimuli/phonemes/ch15-19.TextGrid
sourcedata/stimuli/phonemes/ch20-22.TextGrid
sourcedata/stimuli/phonemes/ch23-25.TextGrid
sourcedata/stimuli/phonemes/ch26-27.TextGrid
sourcedata/stimuli/phonemes/ch4-6.TextGrid
sourcedata/stimuli/phonem

,subject path pattern,subjects containing file,example path
0,sub-XX/ses-01/anat/sub-XX_ses-01_T1w.json,49,sub-01/ses-01/anat/sub-01_ses-01_T1w.json
1,sub-XX/ses-01/anat/sub-XX_ses-01_T1w.nii.gz,49,sub-01/ses-01/anat/sub-01_ses-01_T1w.nii.gz
2,sub-XX/ses-01/meg/sub-XX_ses-01_acq-calibration_meg.dat,58,sub-01/ses-01/meg/sub-01_ses-01_acq-calibration_meg.dat
3,sub-XX/ses-01/meg/sub-XX_ses-01_acq-crosstalk_meg.fif,58,sub-01/ses-01/meg/sub-01_ses-01_acq-crosstalk_meg.fif
4,sub-XX/ses-01/meg/sub-XX_ses-01_coordsystem.json,58,sub-01/ses-01/meg/sub-01_ses-01_coordsystem.json
5,sub-XX/ses-01/meg/sub-XX_ses-01_task-listen_run-01_channels.tsv,58,sub-01/ses-01/meg/sub-01_ses-01_task-listen_run-01_channels.tsv
6,sub-XX/ses-01/meg/sub-XX_ses-01_task-listen_run-01_events.tsv,58,sub-01/ses-01/meg/sub-01_ses-01_task-listen_run-01_events.tsv
7,sub-XX/ses-01/meg/sub-XX_ses-01_task-listen_run-01_meg.fif,58,sub-01/ses-01/meg/sub-01_ses-01_task-listen_run-01_meg.fif
8,sub-XX/ses-01/meg/sub-XX_ses-01_task-listen_run-01_meg.json,58,sub-01/ses-01/meg/sub-01_ses-01_task-listen_run-01_meg.json
9,sub-XX/ses-01/meg/sub-XX_ses-01_task-listen_run-02_channels.tsv,57,sub-01/ses-01/meg/sub-01_ses-01_task-listen_run-02_channels.tsv



Exceptions to the shared subject layout:


,path pattern,missing subjects
0,sub-XX/ses-01/anat/sub-XX_ses-01_T1w.json,"sub-21, sub-22, sub-40, sub-42, sub-43, sub-44, sub-46, sub-48, sub-56"
1,sub-XX/ses-01/anat/sub-XX_ses-01_T1w.nii.gz,"sub-21, sub-22, sub-40, sub-42, sub-43, sub-44, sub-46, sub-48, sub-56"
2,sub-XX/ses-01/meg/sub-XX_ses-01_task-listen_run-02_channels.tsv,sub-56
3,sub-XX/ses-01/meg/sub-XX_ses-01_task-listen_run-02_meg.fif,sub-56
4,sub-XX/ses-01/meg/sub-XX_ses-01_task-listen_run-02_meg.json,sub-56



File and recording coverage by subject:


,subject,files,T1w MRI,complete runs,incomplete runs
0,sub-01,42,True,9,—
1,sub-02,42,True,9,—
2,sub-03,42,True,9,—
3,sub-04,42,True,9,—
4,sub-05,42,True,9,—
5,sub-06,42,True,9,—
6,sub-07,42,True,9,—
7,sub-08,42,True,9,—
8,sub-09,42,True,9,—
9,sub-10,42,True,9,—


In [4]:
import gzip
import struct
import wave


def shorten(value, limit=140):
    text = str(value).replace('\n', ' ')
    return text if len(text) <= limit else text[:limit - 1] + '…'


def file_preview(path):
    size = path.stat().st_size

    if path.suffix.lower() in {'.tsv', '.csv'}:
        separator = '\t' if path.suffix.lower() == '.tsv' else ','
        frame = pd.read_csv(path, sep=separator, nrows=5)
        frame = frame.map(shorten)
        return frame.to_string(index=False).replace('\n', ' | ')

    if path.suffix.lower() == '.json':
        contents = json.dumps(json.loads(path.read_text()), ensure_ascii=False)
        return shorten(contents, 700)

    if path.name.endswith('.nii.gz'):
        with gzip.open(path, 'rb') as file:
            header = file.read(352)
        little_size = struct.unpack('<i', header[:4])[0]
        endian = '<' if little_size == 348 else '>'
        dimensions = struct.unpack(f'{endian}8h', header[40:56])
        voxel_sizes = struct.unpack(f'{endian}8f', header[76:108])
        return (f"NIfTI-1 dimensions={dimensions[1:dimensions[0] + 1]}, "
                f"voxel sizes (mm)={voxel_sizes[1:dimensions[0] + 1]}; "
                f"header bytes={header[:16].hex(' ')}")

    if path.suffix.lower() == '.wav':
        with wave.open(str(path), 'rb') as audio:
            channels = audio.getnchannels()
            sample_width = audio.getsampwidth()
            frames = audio.readframes(4)
            sample_format = {1: 'B', 2: 'h', 4: 'i'}.get(sample_width)
            samples = (
                struct.unpack('<' + sample_format * (len(frames) // sample_width), frames)
                if sample_format else frames.hex(' ')
            )
            return (f"WAV: {channels} channels, {audio.getframerate()} Hz, "
                    f"{audio.getnframes():,} frames, {sample_width} bytes/sample; "
                    f"first samples={samples}")

    if path.suffix.lower() in {'.fif', '.dat'}:
        with path.open('rb') as file:
            first_bytes = file.read(32)
        try:
            text = path.read_text(encoding='utf-8')
            if '\x00' not in text:
                return ' | '.join(shorten(line, 160) for line in text.splitlines()[:5])
        except UnicodeError:
            pass
        return f"Binary data ({size:,} bytes); first 32 bytes={first_bytes.hex(' ')}"

    try:
        lines = path.read_text(encoding='utf-8').splitlines()
        preview = ' | '.join(shorten(line, 180) for line in lines[:5])
        return preview + (f" | … ({len(lines) - 5} more lines)" if len(lines) > 5 else '')
    except UnicodeError:
        with path.open('rb') as file:
            first_bytes = file.read(32)
        return f"Binary data ({size:,} bytes); first 32 bytes={first_bytes.hex(' ')}"


preview_rows = []
for path in all_files:
    try:
        preview = file_preview(path)
    except Exception as error:
        preview = f"Preview error: {type(error).__name__}: {error}"
    preview_rows.append({
        'file': path.relative_to(LISTEN_DIR).as_posix(),
        'size (bytes)': path.stat().st_size,
        'head / content preview': preview,
    })

print(f"Content previews for all {len(preview_rows):,} files under LISTEN_DIR:")
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 1000,
                       'display.width', None):
    display(pd.DataFrame(preview_rows))

KeyboardInterrupt: 

In [ ]:
subjects = sorted(p.name for p in LISTEN_DIR.glob('sub-*'))
print(f"{len(subjects)} subjects: {subjects[0]} ... {subjects[-1]}")

meg_dir = LISTEN_DIR / subjects[0] / 'ses-01' / 'meg'
sorted(p.name for p in meg_dir.iterdir())

58 subjects: sub-01 ... sub-58


['sub-01_ses-01_acq-calibration_meg.dat',
 'sub-01_ses-01_acq-crosstalk_meg.fif',
 'sub-01_ses-01_coordsystem.json',
 'sub-01_ses-01_task-listen_run-01_channels.tsv',
 'sub-01_ses-01_task-listen_run-01_events.tsv',
 'sub-01_ses-01_task-listen_run-01_meg.fif',
 'sub-01_ses-01_task-listen_run-01_meg.json',
 'sub-01_ses-01_task-listen_run-02_channels.tsv',
 'sub-01_ses-01_task-listen_run-02_events.tsv',
 'sub-01_ses-01_task-listen_run-02_meg.fif',
 'sub-01_ses-01_task-listen_run-02_meg.json',
 'sub-01_ses-01_task-listen_run-03_channels.tsv',
 'sub-01_ses-01_task-listen_run-03_events.tsv',
 'sub-01_ses-01_task-listen_run-03_meg.fif',
 'sub-01_ses-01_task-listen_run-03_meg.json',
 'sub-01_ses-01_task-listen_run-04_channels.tsv',
 'sub-01_ses-01_task-listen_run-04_events.tsv',
 'sub-01_ses-01_task-listen_run-04_meg.fif',
 'sub-01_ses-01_task-listen_run-04_meg.json',
 'sub-01_ses-01_task-listen_run-05_channels.tsv',
 'sub-01_ses-01_task-listen_run-05_events.tsv',
 'sub-01_ses-01_task-listen_r